In [ ]:
"""
GitHub Repository Data Processor and LLM Consultant

This script:
1. Fetches markdown documentation from GitHub repositories
2. Processes and chunks the content for efficient search
3. Implements text, vector, and hybrid search capabilities
4. Integrates with OpenAI API for LLM-powered responses

Usage:
    python getfilesrepo.py

Requirements:
    - Set OPENAI_API_KEY environment variable
    - Install dependencies: pip install requests python-frontmatter minsearch sentence-transformers tqdm numpy openai
"""

In [3]:
# need to check if needed on this notebook, my UV might have all of those already

!pip install requests python-frontmatter minsearch sentence-transformers tqdm numpy openai

^C


In [1]:


import os
import io
import zipfile
from typing import List, Dict, Any
import requests
import frontmatter
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm.auto import tqdm
import openai

In [2]:
# OpenAI API Configuration
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("Please set the OPENAI_API_KEY environment variable")

# Model configurations
EMBEDDING_MODEL_NAME = 'multi-qa-distilbert-cos-v1'
LLM_MODEL = 'gpt-4o-mini'

# Search configurations
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 1000
NUM_SEARCH_RESULTS = 5

In [3]:
# =============================================================================
# GITHUB DATA FETCHING
# =============================================================================

def download_github_repo(repo_owner: str, repo_name: str) -> bytes:
    """
    Download a GitHub repository as a ZIP file.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        ZIP file content as bytes

    Raises:
        Exception: If download fails
    """
    url = f'https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main'
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Failed to download repository {repo_owner}/{repo_name}: {response.status_code}")

    return response.content


def parse_markdown_files(zip_content: bytes) -> List[Dict[str, Any]]:
    """
    Parse markdown files from a ZIP archive.

    Args:
        zip_content: ZIP file content as bytes

    Returns:
        List of dictionaries containing parsed file data
    """
    repository_data = []

    with zipfile.ZipFile(io.BytesIO(zip_content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename.lower()

            # Only process markdown files
            if not (filename.endswith('.md') or filename.endswith('.mdx')):
                continue

            try:
                with zf.open(file_info) as f:
                    content = f.read().decode('utf-8', errors='ignore')
                    # Parse frontmatter
                    post = frontmatter.loads(content)
                    data = post.to_dict()
                    data['filename'] = file_info.filename
                    repository_data.append(data)
            except Exception as e:
                print(f"Error processing {file_info.filename}: {e}")
                continue

    return repository_data


def read_repo_data(repo_owner: str, repo_name: str) -> List[Dict[str, Any]]:
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    zip_content = download_github_repo(repo_owner, repo_name)
    return parse_markdown_files(zip_content)

In [6]:
import openai

openai_client = openai.OpenAI()

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
)

print(response.output_text)

Whether you can join a course after it has started usually depends on the course's specific policies. Here are some steps you can take:

1. **Check the Course Details**: Look for any information on late enrollment or prerequisites.
  
2. **Contact the Instructor or Administrator**: Reach out to them to inquire about joining late. 

3. **Review Potential for Catch-Up**: Ask if there are materials available to help you catch up on what you missed.

4. **Look for Similar Courses**: If this one is fully booked or closed, there might be similar options available.

Let me know if you need help with anything specific!


In [ ]:
def text_search(query):
    return faq_index.search(query, num_results=5)


# In[8]:


text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


# In[ ]:


system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)